# 43. Browser Automation — `playwright · selenium`
**Python Libraries Lab · Phase 3 · Web & Networking**

_Drive a real browser: click, type, wait and screenshot._

⚙️ **Runs in Colab** after the setup cell below.

Some pages build their content with JavaScript after they load, so requests and BeautifulSoup see an empty shell. Browser automation tools start a real browser, run the JavaScript, and let your code click buttons, fill forms, wait for results and take screenshots, exactly like a person would. Playwright is the modern choice. Selenium is the long standing standard used in many companies.

**In this notebook:** The notebook serves a small ShopEasy page whose products are drawn by JavaScript. You will show that plain requests cannot see them, then use Playwright to load the page, search, add to cart, read the cart total and take a screenshot. A Selenium version is included to run on your laptop.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Installs Playwright and a headless Chromium browser (about 1 to 2 minutes the first time). Selenium is shown at the end as a script for your laptop.

In [ ]:
!pip install -q playwright
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "Chromium installed"

## Step 1 · A page drawn by JavaScript
The product list only appears after JavaScript runs. Served at port 8079.

In [ ]:
from pathlib import Path
import threading, functools
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

Path("jsapp").mkdir(exist_ok=True)
Path("jsapp/index.html").write_text("""<html><body><h1>ShopEasy</h1>
<input id="q" placeholder="Search products"> <span id="cart">Cart: 0 items, Rs. 0</span>
<ul id="list"></ul>
<script>
const items = [["Steel Bottle",499],["Yoga Mat",1299],["Desk Lamp",899],["Notebook",149],["Water Jug",349]];
let cart = [];
function draw(filter="") {
  setTimeout(() => {                       // simulate data arriving late
    document.getElementById("list").innerHTML = items.filter(i => i[0].toLowerCase().includes(filter))
      .map(i => `<li class="item"><b>${i[0]}</b> Rs. ${i[1]} <button data-name="${i[0]}" data-price="${i[1]}">Add to cart</button></li>`).join("");
    document.querySelectorAll("button").forEach(b => b.onclick = () => {
      cart.push(+b.dataset.price);
      document.getElementById("cart").textContent = `Cart: ${cart.length} items, Rs. ${cart.reduce((a,c)=>a+c,0)}`;
    });
  }, 400);
}
document.getElementById("q").oninput = e => draw(e.target.value.toLowerCase());
draw();
</script></body></html>""")
class Quiet(SimpleHTTPRequestHandler):
    def log_message(self, *a): pass
srv = ThreadingHTTPServer(("127.0.0.1", 8079), functools.partial(Quiet, directory="jsapp"))
threading.Thread(target=srv.serve_forever, daemon=True).start()

import requests
from bs4 import BeautifulSoup
raw = requests.get("http://127.0.0.1:8079/index.html", timeout=5).text
print("Products seen without a browser:", len(BeautifulSoup(raw, "html.parser").select("li.item")))

## Step 2 · Open it in a real (headless) browser
Playwright runs the JavaScript. `wait_for` waits until products appear.

In [ ]:
from playwright.async_api import async_playwright

pw = await async_playwright().start()
browser = await pw.chromium.launch()
page = await browser.new_page(viewport={"width": 800, "height": 500})
await page.goto("http://127.0.0.1:8079/index.html")
await page.locator("li.item").first.wait_for()
names = await page.locator("li.item b").all_inner_texts()
print("Products seen with a browser:", names)

## Step 3 · Type, click and read
Search, add two items to the cart and read the total, like a real user.

In [ ]:
await page.fill("#q", "a")                       # search
await page.wait_for_timeout(600)
print("After search:", await page.locator("li.item b").all_inner_texts())
await page.get_by_role("button", name="Add to cart").first.click()
await page.get_by_role("button", name="Add to cart").nth(1).click()
print(await page.locator("#cart").inner_text())

## Step 4 · Screenshot
See exactly what the browser sees.

In [ ]:
await page.screenshot(path="shop.png")
await browser.close(); await pw.stop()
from IPython.display import Image
Image("shop.png")

## Step 5 · Selenium version for your laptop
Saved as a script. On your laptop: `pip install selenium`, then `python selenium_demo.py` (it downloads a matching Chrome driver by itself). Point it at your own page.

In [ ]:
%%writefile selenium_demo.py
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://books.toscrape.com/"          # a site made for practice

options = webdriver.ChromeOptions()
# options.add_argument("--headless=new")     # uncomment to hide the window
driver = webdriver.Chrome(options=options)
try:
    driver.get(URL)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod")))
    for book in driver.find_elements(By.CSS_SELECTOR, "article.product_pod")[:5]:
        title = book.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title")
        price = book.find_element(By.CSS_SELECTOR, ".price_color").text
        print(f"{title[:40]:<40} {price}")
    driver.save_screenshot("books.png")
finally:
    driver.quit()

## ✍️ Your turn
Start a new browser session, open the page, type `lamp` in the search box, wait 600 ms, and store the **list of product names** shown in `answer`. Close the browser afterwards.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == ["Desk Lamp"], f"Expected ['Desk Lamp'], got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
pw = await async_playwright().start()
browser = await pw.chromium.launch()
page = await browser.new_page()
await page.goto("http://127.0.0.1:8079/index.html")
await page.fill("#q", "lamp")
await page.wait_for_timeout(600)
answer = await page.locator("li.item b").all_inner_texts()
await browser.close(); await pw.stop()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Headless browser**
- **Locator**
- **Auto-wait**
- **Screenshot**
- **WebDriver**
- **Dynamic page**

**Next:** 44. SQLAlchemy ORM (`sqlalchemy`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/44_sqlalchemy.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install playwright selenium requests beautifulsoup4
playwright install chromium
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.